# Lab 14: Path-integral MCMC

We'll sample paths for the quantum harmonic oscillator, then use those paths to recover its position probability density.

The main comparison today: **sampling error versus time-step error**.

PHYS 142/242 · Friday, November 13, 2026

## Set up the notebook

Run the cells in order. Each simulation below starts a new chain from a stated seed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit
from IPython.display import HTML, display

plt.rcParams.update({
    "font.size": 12, "figure.figsize": (6.2, 3.6),
    "figure.dpi": 110, "figure.constrained_layout.use": True,
})
M, OMEGA, HBAR = 1.0, 1.0, 1.0
TAU = 30.0

NumPy holds the paths; Matplotlib draws them. Numba compiles the repeated updates. We use units with $m=\omega=\hbar=1$.

## The result we're aiming for

For the ground state of the harmonic oscillator,

$$\rho_0(x)=|\psi_0(x)|^2=\sqrt{\frac{m\omega}{\pi\hbar}}\,e^{-m\omega x^2/\hbar},
\qquad \langle x^2\rangle_0=\frac{\hbar}{2m\omega}=\frac12.$$

We'll use this known answer to check the simulation.

In [ ]:
x_grid = np.linspace(-3, 3, 301)
rho_ground = np.sqrt(M * OMEGA / (np.pi * HBAR)) * np.exp(
    -M * OMEGA * x_grid**2 / HBAR)
fig, ax = plt.subplots()
ax.plot(x_grid, rho_ground, color="#007c91", lw=2)
ax.set(xlabel="Position x", ylabel="Probability density")
plt.show()

The histogram should approach this curve as the total imaginary time becomes long and the time step becomes small.

## What is one configuration?

**One configuration is a whole curve:** $x=(x_0,x_1,\ldots,x_{N_\tau-1})$.
Each dot is one continuous position at a discrete imaginary time.


In [ ]:
display(HTML('<span class="lab14-visual-marker" hidden></span>'))
lab14_free_rng = np.random.default_rng(1401)
lab14_free_times = np.linspace(0, 4, 41)
lab14_free_dt = lab14_free_times[1]
lab14_free_steps = lab14_free_rng.normal(
    0, np.sqrt(HBAR * lab14_free_dt / M), size=(8, 40))
lab14_free_paths = np.c_[np.zeros(8), np.cumsum(lab14_free_steps, axis=1)]
fig, ax = plt.subplots(figsize=(11.2, 3.0))
for path in lab14_free_paths[1:]:
    ax.plot(lab14_free_times, path, color='#b4c2d0', lw=1.2)
ax.plot(lab14_free_times, lab14_free_paths[0], 'o-', ms=3,
        color='#007c91', lw=2, label='One configuration = this entire path')
point = 23
ax.scatter(lab14_free_times[point], lab14_free_paths[0, point],
           color='#a54632', s=65, zorder=5)
ax.annotate(r'$x_i$: one coordinate',
    (lab14_free_times[point], lab14_free_paths[0, point]),
    xytext=(0.56, 0.12), textcoords='axes fraction',
    arrowprops={'arrowstyle': '->', 'color': '#a54632'}, color='#a54632')
ax.set(xlabel=r'Imaginary time $\tau$', ylabel='Position x', xlim=(0, 4))
ax.legend(loc='upper left', framealpha=0.95, fontsize=11)
plt.show()

Free particle, $V=0$: these examples all start at $x=0$ and have free endpoints.
The gray curves are other possible configurations. **A Monte Carlo update changes a path; it does not advance physical time.**

## Close the path

We impose $x_{N_\tau}=x_0$. The first and last stored points are neighbors.

In [ ]:
dt = 1.0
n_tau = int(round(TAU / dt))
tau_grid = np.arange(n_tau + 1) * dt
example_path = 0.8 * np.sin(2 * np.pi * np.arange(n_tau) / n_tau)
fig, ax = plt.subplots()
ax.plot(tau_grid, np.r_[example_path, example_path[0]], "o-", ms=3)
ax.set(xlabel=r"Imaginary time $\tau$", ylabel="Position x")
plt.show()

The final point repeats the first for drawing only. The array stores $N_\tau$ coordinates, with no duplicate endpoint.

## One closed path, a few, and many

Each curve is a separate configuration. Here we sample free-particle paths with both endpoints fixed at zero.

In [ ]:
display(HTML('<span class="lab14-visual-marker" hidden></span>'))
lab14_bridge_rng = np.random.default_rng(1402)
lab14_bridge_steps = lab14_bridge_rng.normal(
    0, np.sqrt(HBAR * lab14_free_dt / M), size=(100, 40))
lab14_bridge_walks = np.c_[np.zeros(100), np.cumsum(lab14_bridge_steps, axis=1)]
lab14_bridges = lab14_bridge_walks - (
    lab14_free_times / lab14_free_times[-1]) * lab14_bridge_walks[:, -1, None]
fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.0), sharex=True, sharey=True)
for ax, count in zip(axes, [1, 5, 100]):
    for path in lab14_bridges[:count]:
        ax.plot(lab14_free_times, path, color='#7daab8',
                alpha=0.22 if count == 100 else 0.7, lw=1)
    ax.plot(lab14_free_times, lab14_bridges[0], color='#007c91', lw=2)
    ax.scatter([0, 4], [0, 0], color='#a54632', s=30, zorder=5)
    ax.set(title=f'{count} path' + ('s' if count > 1 else ''),
           xlabel=r'Imaginary time $\tau$', xlim=(0, 4))
axes[0].set_ylabel('Position x')
plt.show()

These are **free-particle examples pinned at zero**, not the oscillator samples below.
For the oscillator, we will also sample where the closed path starts.

## Why a long imaginary-time interval?

Closed paths describe thermal equilibrium: $\tau_{\rm total}=\hbar/(k_B T)$.
Longer imaginary time means lower temperature; excited states lose their share of the ensemble.


In [ ]:
display(HTML('<span class="lab14-visual-marker" hidden></span>'))
lab14_b = np.linspace(0.1, 30, 600)
lab14_q = np.exp(-lab14_b)
fig, axes = plt.subplots(1, 2, figsize=(11.2, 3.0))
axes[0].plot(lab14_b, 1-lab14_q, color='#007c91', lw=2, label='Ground state')
axes[0].plot(lab14_b, (1-lab14_q)*lab14_q, color='#a54632', lw=2, label='First excited')
axes[0].plot(lab14_b, lab14_q**2, color='#8a7098', lw=2, label='All higher states')
axes[0].axvline(30, color='#172b4d', ls=':', lw=1.5)
axes[0].set(xlabel=r'Total imaginary time $\omega\tau_{\rm total}$',
            ylabel='Fraction in each state', xlim=(0, 31), ylim=(0, 1.06))
axes[0].legend(loc='center right', fontsize=10)
thermal_x = np.linspace(-3.8, 3.8, 401)
for interval, color in [(1, '#8a7098'), (3, '#a54632'), (30, '#007c91')]:
    variance = HBAR / (2*M*OMEGA*np.tanh(interval/2))
    density = np.exp(-thermal_x**2/(2*variance))/np.sqrt(2*np.pi*variance)
    axes[1].plot(thermal_x, density, color=color, lw=2,
                 label=rf'$\omega\tau_{{\rm total}}={interval}$')
axes[1].plot(thermal_x, np.sqrt(M*OMEGA/(np.pi*HBAR))*
             np.exp(-M*OMEGA*thermal_x**2/HBAR), '--', color='#172b4d',
             lw=1.5, label='Ground state')
axes[1].set(xlabel='Position x', ylabel='Position density', xlim=(-3.8, 3.8))
axes[1].legend(fontsize=10)
plt.show()

**Continuum oscillator predictions.** At $\omega\tau_{\rm total}=30$, the ground state dominates.
A long interval does **not** remove finite-$\Delta\tau$ error; we test that later.

## One link: the equation and its code

$$s(x_i,x_{i+1})=\underbrace{\frac{m(x_{i+1}-x_i)^2}{2\Delta\tau}}_{\text{kinetic: penalizes jumps}}+
\underbrace{\Delta\tau\,\frac{m\omega^2x_i^2}{2}}_{\text{potential: penalizes large positions}}.$$


In [ ]:
@njit
def action(x_left, x_right, dt):
    kinetic = 0.5 * M * (x_right - x_left)**2 / dt
    potential = 0.5 * M * OMEGA**2 * x_left**2 * dt
    return kinetic + potential

The potential uses the **left endpoint**, as in Assignment 3 and the corrected course lab.

## The whole path: action and statistical weight

$$S_E(x)=\sum_{i=0}^{N_\tau-1}s(x_i,x_{i+1}),\qquad x_{N_\tau}=x_0,
\qquad P(x)\propto e^{-S_E(x)/\hbar}.$$


In [ ]:
@njit
def total_action(path, dt):
    result = 0.0
    for i in range(path.size):
        result += action(path[i], path[(i + 1) % path.size], dt)
    return result

The modulo operation closes the last link. Each link and potential term are counted once.
**Larger action means smaller statistical weight for that path.**

## Watch the action change

Move one point, then shift the whole path. Watch the jump cost, the potential cost, and the weight change.

In [ ]:
display(HTML('<span class="lab14-visual-marker" hidden></span>'))
import numpy as lab14_np
import re as lab14_re
import matplotlib.pyplot as lab14_plt
from matplotlib.animation import FuncAnimation as lab14_FuncAnimation
from IPython.display import HTML as lab14_HTML, display as lab14_display

# This illustration uses its own path and does not advance the MCMC chain.
lab14_action_m = lab14_action_omega = lab14_action_hbar = 1.0
lab14_action_dt = 1.0
lab14_action_n = 30
lab14_action_index = lab14_action_n // 2
lab14_action_tau = lab14_np.arange(lab14_action_n + 1) * lab14_action_dt
lab14_action_reference_path = 0.35 * lab14_np.sin(
    2 * lab14_np.pi * lab14_np.arange(lab14_action_n) / lab14_action_n)
lab14_action_reference_path -= lab14_action_reference_path.mean()


def lab14_action_parts(path):
    """Kinetic and potential contributions for the periodic oscillator path."""
    differences = lab14_np.roll(path, -1) - path
    kinetic = lab14_action_m * lab14_np.sum(differences**2) / (2 * lab14_action_dt)
    potential = (lab14_action_dt * lab14_action_m * lab14_action_omega**2
                 * lab14_np.sum(path**2) / 2)
    return kinetic, potential


# Both phases return to, and are compared with, this same fixed reference.
lab14_action_reference_parts = lab14_np.array(
    lab14_action_parts(lab14_action_reference_path))
lab14_action_reference_total = lab14_action_reference_parts.sum()
lab14_action_profile = lab14_np.sin(lab14_np.linspace(0, lab14_np.pi, 25))
lab14_action_profile[[0, -1]] = 0.0
lab14_action_offsets = lab14_np.r_[1.1 * lab14_action_profile,
                                0.35 * lab14_action_profile]
lab14_action_phase_names = (['Move one point'] * 25
                           + ['Shift the whole path'] * 25)
lab14_action_paths = lab14_np.tile(lab14_action_reference_path, (50, 1))
lab14_action_paths[:25, lab14_action_index] += lab14_action_offsets[:25]
lab14_action_paths[25:] += lab14_action_offsets[25:, None]
lab14_action_components = lab14_np.array(
    [lab14_action_parts(path) for path in lab14_action_paths])
lab14_action_kinetic = lab14_action_components[:, 0]
lab14_action_potential = lab14_action_components[:, 1]
lab14_action_total = lab14_action_components.sum(axis=1)
lab14_action_relative_weight = lab14_np.exp(
    -(lab14_action_total - lab14_action_reference_total) / lab14_action_hbar)

lab14_action_figure = lab14_plt.figure(figsize=(10, 2.8), dpi=100, layout='none')
lab14_action_grid = lab14_action_figure.add_gridspec(
    2, 2, width_ratios=(1.85, 1), height_ratios=(3.2, 1),
    left=0.075, right=0.98, bottom=0.18, top=0.88, wspace=0.34, hspace=0.65)
lab14_action_path_ax = lab14_action_figure.add_subplot(lab14_action_grid[:, 0])
lab14_action_bar_ax = lab14_action_figure.add_subplot(lab14_action_grid[0, 1])
lab14_action_weight_ax = lab14_action_figure.add_subplot(lab14_action_grid[1, 1])
lab14_action_weight_ax.set_axis_off()
lab14_action_closed_reference = lab14_np.r_[lab14_action_reference_path,
                                           lab14_action_reference_path[0]]
lab14_action_path_ax.plot(
    lab14_action_tau, lab14_action_closed_reference, '--',
    color='#707070', lw=2, label='Fixed reference', zorder=1)
lab14_action_line, = lab14_action_path_ax.plot(
    lab14_action_tau, lab14_action_closed_reference, 'o-',
    color='#007c91', ms=3, lw=2, label='Current path', zorder=2)
lab14_action_link_line, = lab14_action_path_ax.plot(
    [], [], 'o-', color='#b84b20', ms=5, lw=3, zorder=3)
lab14_action_path_ax.set(
    xlim=(0, lab14_action_tau[-1]),
    ylim=(lab14_action_paths.min() - 0.18, lab14_action_paths.max() + 0.18),
    xlabel='Imaginary time τ', ylabel='Position x')
lab14_action_title = lab14_action_path_ax.set_title('Move one point', fontsize=14)
lab14_action_path_ax.legend(loc='upper left', fontsize=12, frameon=False)
lab14_action_bars = lab14_action_bar_ax.bar(
    [0, 1], lab14_action_reference_parts / lab14_action_hbar,
    color=['#007c91', '#b84b20'], width=0.55)
for lab14_action_bar_index, lab14_action_ref_value in enumerate(
        lab14_action_reference_parts / lab14_action_hbar):
    lab14_action_bar_ax.plot(
        [lab14_action_bar_index - 0.36, lab14_action_bar_index + 0.36],
        [lab14_action_ref_value] * 2, '--', color='#707070', lw=1.5)
lab14_action_bar_ax.set(
    xticks=[0, 1], xticklabels=['Kinetic / ħ', 'Potential / ħ'],
    ylim=(0, 1.16 * lab14_action_components.max() / lab14_action_hbar),
    ylabel='Action')
lab14_action_bar_ax.set_title('Action contributions', fontsize=12)
lab14_action_bar_ax.set_yticks([0, 1, 2, 3])
lab14_action_weight_text = lab14_action_weight_ax.text(
    0.5, 0.75, '', transform=lab14_action_weight_ax.transAxes,
    ha='center', va='center', fontsize=13)
lab14_action_weight_ax.text(
    0.5, 0.05, r'$w/w_{\rm ref}=e^{-(S-S_{\rm ref})/\hbar}$',
    transform=lab14_action_weight_ax.transAxes,
    ha='center', va='center', fontsize=12)
for lab14_action_axis in (lab14_action_path_ax, lab14_action_bar_ax):
    lab14_action_axis.tick_params(labelsize=12)
    lab14_action_axis.xaxis.label.set_size(12)
    lab14_action_axis.yaxis.label.set_size(12)


def lab14_action_update(frame):
    """Draw one precomputed deformation without touching sampler state."""
    path = lab14_action_paths[frame]
    lab14_action_line.set_ydata(lab14_np.r_[path, path[0]])
    lab14_action_title.set_text(lab14_action_phase_names[frame])
    if frame < 25:
        nearby = lab14_np.arange(lab14_action_index - 1, lab14_action_index + 2)
        lab14_action_link_line.set_data(nearby * lab14_action_dt, path[nearby])
    else:
        lab14_action_link_line.set_data([], [])
    for bar, contribution in zip(lab14_action_bars, lab14_action_components[frame]):
        bar.set_height(contribution / lab14_action_hbar)
    lab14_action_weight_text.set_text(
        f'Relative weight = {lab14_action_relative_weight[frame]:.3f}')
    return (lab14_action_line, lab14_action_link_line, lab14_action_title,
            lab14_action_weight_text, *lab14_action_bars)


lab14_action_update(0)
lab14_action_animation = lab14_FuncAnimation(
    lab14_action_figure, lab14_action_update, frames=50,
    interval=110, repeat=False, blit=False)
# JSHTML starts paused. "Once" is the selected playback mode.
# Frame export can re-read layout defaults; preserve this explicit GridSpec.
with lab14_plt.rc_context({'figure.constrained_layout.use': False, 'figure.autolayout': False}):
    lab14_action_animation_html = lab14_action_animation.to_jshtml(default_mode='once')
# Matplotlib adds a remote icon stylesheet; native symbols keep this output offline.
lab14_action_animation_html = lab14_re.sub(
    r'<link\b[^>]*>', '', lab14_action_animation_html)
lab14_action_control_symbols = {
    'fa fa-fast-backward': '⏮', 'fa fa-step-backward': '‹',
    'fa fa-play fa-flip-horizontal': '◀', 'fa fa-pause': 'Ⅱ',
    'fa fa-play': '▶', 'fa fa-step-forward': '›',
    'fa fa-fast-forward': '⏭', 'fa fa-minus': '−', 'fa fa-plus': '+',
}
for lab14_action_icon, lab14_action_symbol in lab14_action_control_symbols.items():
    lab14_action_animation_html = lab14_action_animation_html.replace(
        f'<i class="{lab14_action_icon}"></i>',
        f'<span style="font-size:16px">{lab14_action_symbol}</span>')
lab14_plt.close(lab14_action_figure)
lab14_display(lab14_HTML(
    "<div class='lab14-path-animation'>"
    "<style>.lab14-path-animation .anim-buttons{margin:3px 0}"
    ".lab14-path-animation .anim-state{font-size:12px}"
    ".lab14-path-animation .anim-buttons button{height:22px}</style>"
    + lab14_action_animation_html
    + "<p style='font-size:14px;margin:3px 0'>Dashed marks show the fixed reference; "
      "orange highlights the two affected links.</p></div>"))

Controlled deformations, not a Monte Carlo run. The displayed weight is **relative to the reference path**, $e^{-(S_E-S_{\rm ref})/\hbar}$; it is not a normalized probability.

## Change one point

Propose $x_i'=x_i+u$, with $u$ uniform on $[-h,h]$.

Only the two links touching $i$ change. We can compute $\Delta S_E$ without summing the entire path.

In [ ]:
i = n_tau // 2
proposal = example_path.copy()
proposal[i] += 0.65
fig, ax = plt.subplots()
ax.plot(tau_grid, np.r_[example_path, example_path[0]], 'o-',
        color='#172b4d', label='Current path')
ax.plot(tau_grid, np.r_[proposal, proposal[0]], 'o--',
        color='#a54632', label='Proposed path')
ax.plot(tau_grid[i-1:i+2], proposal[i-1:i+2], color='#a54632', lw=3)
ax.annotate('One coordinate changes', (tau_grid[i], proposal[i]),
            xytext=(i*dt+0.3, proposal[i]+0.18),
            arrowprops={'arrowstyle': '->'}, fontsize=10)
ax.set(xlim=((i - 3) * dt, (i + 3) * dt),
       ylim=(-0.65, 1.05), xlabel=r'Imaginary time $\tau$', ylabel='Position x')
ax.legend(loc='lower left', fontsize=10)
plt.show()

Everything outside those two links cancels when we subtract the old action from the proposed action.

## Compute the local action change

Replace the two old links by the two proposed links.

In [ ]:
@njit
def delta_action(path, x_prime, i, dt):
    left = path[(i - 1) % path.size]
    right = path[(i + 1) % path.size]
    new = action(left, x_prime, dt) + action(x_prime, right, dt)
    old = action(left, path[i], dt) + action(path[i], right, dt)
    return new - old

This works at $i=0$ and at the last stored point because both neighbor indices wrap around.

## Check the local calculation

Recompute the full action once and compare. This catches missing links and periodic-boundary mistakes.

In [ ]:
local_change = delta_action(example_path, proposal[i], i, dt)
full_change = total_action(proposal, dt) - total_action(example_path, dt)
print(f"Local change: {local_change:.8f}")
print(f"Full change:  {full_change:.8f}")
np.testing.assert_allclose(local_change, full_change, atol=1e-12)

Agreement here checks the action calculation. It does not yet show that a Markov chain has equilibrated.

## Accept or reject

The proposal is symmetric, so the Metropolis acceptance probability is

$$A=\min\left(1,e^{-\Delta S_E/\hbar}\right).$$

Accept every move that lowers the action. Sometimes accept a move that raises it. A rejected move leaves the path unchanged and still counts as an attempt.

## One Metropolis update

The oscillator has no hard position walls. We let the path explore the real line.

In [ ]:
@njit
def update_point(path, i, dt, hitsize, rng):
    x_prime = path[i] + rng.uniform(-hitsize, hitsize)
    change = delta_action(path, x_prime, i, dt)
    if change <= 0 or rng.random() < np.exp(-change / HBAR):
        path[i] = x_prime
        return 1
    return 0

Plot limits only control the picture. Redrawing a proposal until it lies inside the plotting window would change its probability near the edges.

## One sweep

Visit every stored point once, in order. Each accepted change is immediately used by the following update.

In [ ]:
@njit
def sweep(path, dt, hitsize, rng):
    accepted = 0
    for i in range(path.size):
        accepted += update_point(path, i, dt, hitsize, rng)
    return accepted / path.size

In [ ]:
display(HTML('<span class="lab14-visual-marker" hidden></span>'))
lab14_sweep_rng = np.random.default_rng(1403)
lab14_before_sweep = example_path.copy()
lab14_after_one = lab14_before_sweep.copy()
lab14_first_accepted = update_point(lab14_after_one, 0, dt, 1.0, lab14_sweep_rng)
lab14_after_sweep = lab14_after_one.copy()
lab14_sweep_accepts = lab14_first_accepted
for point in range(1, n_tau):
    lab14_sweep_accepts += update_point(lab14_after_sweep, point, dt, 1.0, lab14_sweep_rng)
fig, axes = plt.subplots(1, 3, figsize=(11.2, 2.25), sharex=True, sharey=True)
stages = [(lab14_before_sweep, 'Starting path'),
          (lab14_after_one, 'After 1 attempt: ' + ('accepted' if lab14_first_accepted else 'rejected')),
          (lab14_after_sweep, f'After 1 sweep: {lab14_sweep_accepts}/{n_tau} accepted')]
for ax, (path, title) in zip(axes, stages):
    ax.plot(tau_grid, np.r_[lab14_before_sweep, lab14_before_sweep[0]],
            color='#b4c2d0', lw=1.5)
    ax.plot(tau_grid, np.r_[path, path[0]], 'o-', color='#007c91', ms=2.5)
    ax.set(title=title, xlabel=r'Imaginary time $\tau$')
axes[0].set_ylabel('Position x')
plt.show()

Gray shows the starting path. One attempt can change one coordinate; one sweep attempts every coordinate.
**A retained sample records the whole resulting path**, including any coordinates that did not change.

## Warm up, then collect measurements

Start from zeros. Keep a trace of the warm-up, but exclude it from the histogram and reported average.

In [ ]:
@njit
def sample_paths(n_tau, dt, hitsize, burn_in, sweeps, rng):
    path = np.zeros(n_tau)
    warmup = np.empty(burn_in)
    samples = np.empty((sweeps, n_tau))
    acceptance = np.empty(sweeps)
    for k in range(burn_in):
        sweep(path, dt, hitsize, rng)
        warmup[k] = np.mean(path**2)
    for k in range(sweeps):
        acceptance[k] = sweep(path, dt, hitsize, rng)
        samples[k] = path
    return warmup, samples, acceptance

Each row of `samples` is one retained path. Copying it into a row preserves that configuration before the next sweep changes `path`.

## Run the coarse time grid

Begin with the original lab's $\Delta\tau=1$ and total imaginary time 30. A wider proposal helps the path explore faster than the original $h=0.1$.

In [ ]:
DT = 1.0
NTAU = int(round(TAU / DT))
HITSIZE = 1.0
BURN_IN = 10_000
SWEEPS = 40_000
warmup, samples, acceptance = sample_paths(
    NTAU, DT, HITSIZE, BURN_IN, SWEEPS, np.random.default_rng(142))
x2_trace = np.mean(samples**2, axis=1)
print(f"{samples.shape[0]:,} paths; {samples.shape[1]} points per path")
print(f"Accepted fraction: {acceptance.mean():.3f}")

The first call includes compilation. Rerunning this cell restarts the chain from the same seed; it does not accumulate old measurements.

## Watch the chain

For each path, measure $\overline{x^2}=N_\tau^{-1}\sum_i x_i^2$. This is one measurement per sweep.

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(BURN_IN), warmup, lw=0.5, color="#a54632", label="Discarded")
ax.plot(BURN_IN + np.arange(SWEEPS), x2_trace,
        lw=0.5, color="#007c91", label="Retained")
ax.axvline(BURN_IN, color="black", ls="--", lw=1)
ax.set(xlabel="Sweep", ylabel=r"Path mean of $x^2$")
ax.legend()
plt.show()

Look for drift and long correlated stretches. A trace is a useful diagnostic, not proof of equilibrium; production work should compare starts and run lengths.

## One sample, a few samples, many samples

Top: actual oscillator paths saved by our MCMC run. Bottom: the positions collected from those same paths.

In [ ]:
display(HTML('<span class="lab14-visual-marker" hidden></span>'))
lab14_counts = [1, 5, len(samples)]
lab14_limit = np.ceil(np.abs(samples).max()*2)/2
lab14_bins = np.linspace(-lab14_limit, lab14_limit, 41)
fig, axes = plt.subplots(2, 3, figsize=(11.2, 3.8), sharex='row', sharey='row')
for col, count in enumerate(lab14_counts):
    retained = samples[:count]
    drawn = np.linspace(0, count-1, min(count, 100), dtype=int)
    for index in drawn:
        axes[0, col].plot(np.arange(NTAU+1)*DT,
            np.r_[retained[index], retained[index, 0]],
            color='#007c91', lw=1, alpha=0.15 if count > 100 else 0.7)
    title = f'{count:,} saved path' + ('s' if count > 1 else '')
    if count > 100:
        title += '\n100 drawn; all used below'
    axes[0, col].set(title=title, xlabel=r'Imaginary time $\tau$',
                     ylim=(-lab14_limit, lab14_limit))
    axes[1, col].hist(retained.ravel(), bins=lab14_bins, density=True,
                     color='#9fc9d3', edgecolor='white', linewidth=0.4)
    axes[1, col].set(xlabel='Position x', xlim=(-lab14_limit, lab14_limit))
    axes[1, col].text(0.97, 0.91, f'{retained.size:,} coordinates',
                     transform=axes[1, col].transAxes, ha='right', va='top', fontsize=10)
axes[0, 0].set_ylabel('Position x')
axes[1, 0].set_ylabel('Position density')
plt.show()

**One coordinate is not one path sample.** More sampled paths fill out the position distribution.
Coordinates within a path and successive saved paths are correlated; we account for that in the uncertainty.

## The finite grid has its own answer

For a long harmonic-oscillator path with this left-endpoint action,

$$\langle x^2\rangle_{\Delta\tau}\simeq
\frac{\hbar}{m\omega\sqrt{4+(\omega\Delta\tau)^2}}.$$

At $\Delta\tau=1$, this is **0.4472**, not 0.5. The exact finite-path result can also be found by summing its normal modes.

In [ ]:
def lattice_variance(n_tau, dt):
    modes = np.arange(n_tau)
    eigenvalues = dt * OMEGA**2 + 4 / dt * np.sin(np.pi * modes / n_tau)**2
    return HBAR / M * np.mean(1 / eigenvalues)

variance_dt = lattice_variance(NTAU, DT)
print(f"Finite-grid prediction: {variance_dt:.6f}")

This is a reference calculation, not part of the sampler. Keeping the left-endpoint potential makes the time-step error visible.

## Compare probability densities

First compare with the distribution the finite grid actually samples. Then compare with the continuum ground state.

In [ ]:
rho_lattice = np.exp(-x_grid**2 / (2 * variance_dt)) / np.sqrt(
    2 * np.pi * variance_dt)
fig, ax = plt.subplots()
ax.hist(samples.ravel(), bins=100, density=True, color="#9fc9d3", label="MCMC")
ax.plot(x_grid, rho_lattice, color="#172b4d", lw=2, label="Finite grid")
ax.plot(x_grid, rho_ground, "--", color="#a54632", lw=2, label="Continuum")
ax.set(xlim=(-3, 3), xlabel="Position x", ylabel="Probability density")
ax.legend()
plt.show()

A narrower histogram at this time step is expected. More sweeps reduce sampling noise; they do not remove this discretization error.

## Put an uncertainty on the width

Use the per-sweep measurements and the blocking method from Lab 12. Neighboring blocks must be long enough to be approximately independent.

In [ ]:
def blocked_mean_error(values, block_size):
    n_blocks = values.size // block_size
    blocks = values[:n_blocks * block_size].reshape(n_blocks, block_size)
    means = blocks.mean(axis=1)
    return values.mean(), means.std(ddof=1) / np.sqrt(n_blocks)

for block_size in (250, 500, 1000, 2000):
    mean, error = blocked_mean_error(x2_trace, block_size)
    print(f"Block {block_size:4d}: <x²> = {mean:.4f} ± {error:.4f}")

Look for an error estimate that settles as the block size grows, while retaining enough blocks. If it keeps growing, collect a longer chain.

## Reduce the time step

Keep the total imaginary time fixed. Halving the step doubles the number of coordinates in each path.

In [ ]:
time_steps = np.array([1.0, 0.5, 0.25])
means, errors = [blocked_mean_error(x2_trace, 1000)[0]], [
    blocked_mean_error(x2_trace, 1000)[1]]
for run, smaller_dt in enumerate(time_steps[1:], start=1):
    smaller_n = int(round(TAU / smaller_dt))
    _, fine_samples, _ = sample_paths(
        smaller_n, smaller_dt, HITSIZE * np.sqrt(smaller_dt),
        BURN_IN, SWEEPS, np.random.default_rng(142 + run))
    mean, error = blocked_mean_error(np.mean(fine_samples**2, axis=1), 1000)
    means.append(mean)
    errors.append(error)

Scaling the proposal with $\sqrt{\Delta\tau}$ keeps the kinetic-action change comparable. We still need to check mixing as the time grid gets finer.

## Separate the two errors

The error bars estimate sampling uncertainty. The curve shows the bias from the finite time step.

In [ ]:
dt_curve = np.linspace(0, 1.05, 200)
variance_curve = HBAR / (M * OMEGA * np.sqrt(4 + (OMEGA * dt_curve)**2))
fig, ax = plt.subplots()
ax.plot(dt_curve, variance_curve, color="#172b4d", label="Long-path grid prediction")
ax.axhline(0.5, color="#a54632", ls="--", label="Continuum ground state")
ax.errorbar(time_steps, means, yerr=errors, fmt="o", capsize=4,
            color="#007c91", label="MCMC ± blocked error")
ax.set(xlabel=r"Time step $\Delta\tau$", ylabel=r"$\langle x^2\rangle$")
ax.legend(fontsize=10)
plt.show()

Agreement with the grid prediction checks the sampler. Convergence toward 0.5 checks the time discretization. The finer chain is more correlated, so its blocked error estimate is noisier.

## What the run controls do

| Change | What it addresses |
|---|---|
| More retained sweeps | Sampling uncertainty, once equilibrated |
| Longer discarded warm-up | Sensitivity to the starting path |
| Different proposal size | Acceptance and exploration speed |
| Smaller $\Delta\tau$ at fixed total time | Time-discretization error |
| Longer total imaginary time | Excited-state contributions |

An acceptance fraction alone does not tell us whether the chain explores well.

## Carry this into Assignment 3

Follow the assignment's specified total imaginary time and action. At finite total time, compare with the thermal result, not automatically with the ground state:

$$\langle x^2\rangle=\frac{\hbar}{2m\omega}\coth\!\left(\frac{\omega\tau_{\rm total}}2\right).$$

For each result, state the time step, run length, discarded warm-up, and uncertainty method. Check the sampler against the finite grid before interpreting a continuum discrepancy.

## The workflow we used

1. Define the path and its action.
2. Check a local action change against a full recomputation.
3. Sample with symmetric proposals and a stated warm-up.
4. Measure the density and $\langle x^2\rangle$ with correlated errors.
5. Reduce the time step and compare with the known answer.

Adapted from Javier Duarte's Lab 14 and the Mittal et al. example discussed in lecture. The left-endpoint potential follows the corrected course notebook.